# Grouped Query Attention (GQA)

GQA 是 MHA 和 MQA 之间的折中方案，在保持性能的同时减少 KV Cache 的内存占用。

## 三种 Attention 变体对比

| 类型 | Q 头数 | K 头数 | V 头数 | 特点 |
|------|--------|--------|--------|------|
| MHA  | n      | n      | n      | 每个 Q 头独立的 KV 头 |
| MQA  | n      | 1      | 1      | 所有 Q 头共享 1 个 KV 头 |
| GQA  | n      | n/g    | n/g    | 每 g 个 Q 头共享 1 个 KV 头 |

## 为什么需要 GQA？

- **MHA**: 质量最好，但 KV Cache 占用大
- **MQA**: KV Cache 最小，但质量下降
- **GQA**: 平衡质量和内存，是工业界的主流选择（LLaMA 2、Mistral 等）

In [1]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from src.gqa import GroupedQueryAttention, compare_attention_variants
from src.attention import MultiHeadAttention

torch.manual_seed(42)

## 1. GQA 配置示例

In [2]:
# 常见的 GQA 配置
print("常见的 Attention 配置:\n")
print(f"{'名称':<10} {'Q 头数':<10} {'KV 头数':<10} {'分组比例':<10} {'示例模型'}")
print("-" * 70)
print(f"{'MHA':<10} {32:<10} {32:<10} {'1:1':<10} {'GPT-3, BERT'}")
print(f"{'GQA-8':<10} {32:<10} {8:<10} {'4:1':<10} {'LLaMA 2 (70B)'}")
print(f"{'GQA-4':<10} {32:<10} {4:<10} {'8:1':<10} {'Mistral'}")
print(f"{'MQA':<10} {32:<10} {1:<10} {'32:1':<10} {'PaLM'}")

常见的 Attention 配置:

名称         Q 头数       KV 头数      分组比例       示例模型
----------------------------------------------------------------------
MHA        32         32         1:1        GPT-3, BERT
GQA-8      32         8          4:1        LLaMA 2 (70B)
GQA-4      32         4          8:1        Mistral
MQA        32         1          32:1       PaLM


## 2. 创建和测试 GQA

In [3]:
# 配置
batch_size = 2
seq_len = 10
d_model = 512
num_q_heads = 32
num_kv_heads = 8  # 每 4 个 Q 头共享 1 个 KV 头

# 创建 GQA
gqa = GroupedQueryAttention(d_model, num_q_heads, num_kv_heads)

print("GQA 配置:")
print(gqa)

# 测试
x = torch.randn(batch_size, seq_len, d_model)
output, attn_weights = gqa(x, x, x)

print(f"\n输入形状: {x.shape}")
print(f"输出形状: {output.shape}")
print(f"注意力权重形状: {attn_weights.shape}")

GQA 配置:
GroupedQueryAttention(
  d_model=512, num_q_heads=32, num_kv_heads=8, num_groups=4, d_k=16
  (W_q): Linear(in_features=512, out_features=512, bias=True)
  (W_k): Linear(in_features=512, out_features=128, bias=True)
  (W_v): Linear(in_features=512, out_features=128, bias=True)
  (W_o): Linear(in_features=512, out_features=512, bias=True)
)

输入形状: torch.Size([2, 10, 512])
输出形状: torch.Size([2, 10, 512])
注意力权重形状: torch.Size([2, 32, 10, 10])


## 3. 参数量和内存对比

In [4]:
# 使用预定义的比较函数
compare_attention_variants(d_model=4096, num_heads=32, seq_len=2048)

配置: d_model=4096, num_heads=32, seq_len=2048

MHA (Multi-Head Attention):
  - Q/K/V 头数: 32/32/32
  - 参数量: 67,108,864 (67.11M)
  - FLOPs: 68,719,476,736 (68.72G)

MQA (Multi-Query Attention):
  - Q/K/V 头数: 32/1/1
  - 参数量: 34,603,008 (34.60M)
  - 参数减少: 48.4%
  - FLOPs: 68,719,476,736 (68.72G)

GQA (Grouped Query Attention, 4:1):
  - Q/K/V 头数: 32/8/8
  - 参数量: 41,943,040 (41.94M)
  - 参数减少: 37.5%
  - FLOPs: 68,719,476,736 (68.72G)



## 4. KV Cache 内存占用对比

In [5]:
def calculate_kv_cache_size(batch_size, seq_len, num_kv_heads, head_dim, dtype_bytes=2):
    """计算 KV Cache 的内存占用 (MB)"""
    size_bytes = 2 * batch_size * seq_len * num_kv_heads * head_dim * dtype_bytes
    return size_bytes / (1024 ** 2)

# 配置
batch_size = 32
seq_len = 2048
num_q_heads = 32
head_dim = 128

configs = [
    ("MHA", 32),
    ("GQA-8", 8),
    ("GQA-4", 4),
    ("MQA", 1),
]

print("KV Cache 内存占用对比 (FP16):\n")
print(f"配置: batch_size={batch_size}, seq_len={seq_len}, head_dim={head_dim}\n")
print(f"{'类型':<10} {'KV 头数':<10} {'内存 (MB)':<15} {'相对 MHA'}")
print("-" * 50)

mha_size = None
sizes = []
for name, num_kv_heads in configs:
    size = calculate_kv_cache_size(batch_size, seq_len, num_kv_heads, head_dim)
    sizes.append(size)
    
    if mha_size is None:
        mha_size = size
        ratio = "100%"
    else:
        ratio = f"{size / mha_size * 100:.1f}%"
    
    print(f"{name:<10} {num_kv_heads:<10} {size:<15.2f} {ratio}")

print("\n关键观察:")
print(f"- GQA-8 相比 MHA 节省 {(1 - sizes[1]/sizes[0]) * 100:.0f}% 内存")
print(f"- GQA-4 相比 MHA 节省 {(1 - sizes[2]/sizes[0]) * 100:.0f}% 内存")
print(f"- MQA 相比 MHA 节省 {(1 - sizes[3]/sizes[0]) * 100:.0f}% 内存")

KV Cache 内存占用对比 (FP16):

配置: batch_size=32, seq_len=2048, head_dim=128

类型         KV 头数      内存 (MB)         相对 MHA
--------------------------------------------------
MHA        32         1024.00         100%
GQA-8      8          256.00          25.0%
GQA-4      4          128.00          12.5%
MQA        1          32.00           3.1%

关键观察:
- GQA-8 相比 MHA 节省 75% 内存
- GQA-4 相比 MHA 节省 88% 内存
- MQA 相比 MHA 节省 97% 内存


## 5. 总结

### GQA 的关键优势：

1. **内存效率**: 显著减少 KV Cache 占用
2. **质量保持**: 相比 MQA，质量损失很小
3. **推理加速**: 减少内存访问，提升速度
4. **工业验证**: LLaMA 2、Mistral 等主流模型采用

### 下一步：
- `04_kv_cache.ipynb`: 学习 KV Cache 的实现和优化